In [ ]:
---
toc: false 
layout: post
title: Fibonacci Jumper
description: An AP CSA game teaching arrays, loops and classes through Fibonacci.
courses: { csa: {week: 25} }
type: ccc
image: /images/data_structures/fibonacci.png
permalink: /fibonacci
---


![game](https://upload.wikimedia.org/wikipedia/commons/thumb/1/15/Fibonacci_Squares.svg/1200px-Fibonacci_Squares.svg.png)

## Fibonacci Jumper Game

Welcome to **Fibonacci Jumper**! A game where you control a character, and each jump must be the next Fibonacci number.

### AP CSA Concepts Taught:
- **Arrays/ArrayLists**: Tracking our Fibonacci sequence and jumps.
- **Loops**: Generating the jumps up to N limits.
- **Conditionals**: Win/Lose condition logic based on overshoot.
- **Methods**: Moving the character incrementally.
- **Classes**: Structuring the Game and Player logic.


In [ ]:
%%js

// GAME_RUNNER: Fibonacci Jumper overview with stations. | hide_edit: true

import { GameControl, GameEnvBackground, Player, NPC } from '/assets/js/GameEnginev1.1/essentials/Imports.js';
import Barrier from '/assets/js/GameEnginev1.1/essentials/Barrier.js';

class JumperPlayer extends Player {
  constructor(data, gameEnv) {
    super(data, gameEnv);

    // Core game rules
    this.fiboSeq = [1, 1, 2, 3, 5, 8, 13, 21, 34];
    this.jumpIndex = 0;
    this.units = 0;
    this.targetUnits = 33;

    // Bounds to keep it game-like (you can tune these)
    this.safeMin = -5;
    this.safeMax = this.targetUnits + 10;

    // Convert "units" to pixels along x
    this.startX = this.position.x;
    this.scalar = gameEnv.innerWidth * 0.02;

    // UI + input guards
    this.done = false;
    this._lastKeyDown = null;
    this._hud = null;
    this._modal = null;

    this.initHud();
    this.updateHud('Press A/D to jump Fibonacci numbers. Reach 33 exactly to win.');
  }

  // Disable continuous movement; only discrete Fibonacci jumps
  updateVelocity() {
    this.velocity.x = 0;
    this.velocity.y = 0;
  }

  initHud() {
    const container = this.gameEnv && this.gameEnv.container ? this.gameEnv.container : null;
    if (!container) return;

    // Prevent duplicates if engine reuses container
    const existing = container.querySelector('.fj-hud');
    if (existing) {
      this._hud = existing;
      return;
    }

    const hud = document.createElement('div');
    hud.className = 'fj-hud';
    hud.style.position = 'absolute';
    hud.style.left = '10px';
    hud.style.top = '10px';
    hud.style.zIndex = '9999';
    hud.style.padding = '10px 12px';
    hud.style.borderRadius = '12px';
    hud.style.color = '#EAF0FF';
    hud.style.background = 'rgba(0,0,0,0.55)';
    hud.style.border = '1px solid rgba(255,255,255,0.18)';
    hud.style.fontFamily = 'ui-sans-serif, system-ui, -apple-system, Segoe UI, sans-serif';
    hud.style.fontSize = '12px';
    hud.style.lineHeight = '1.35';
    hud.style.maxWidth = '360px';

    const title = document.createElement('div');
    title.style.fontWeight = '800';
    title.style.marginBottom = '6px';
    title.textContent = 'Fibonacci Jumper';

    const stats = document.createElement('div');
    stats.className = 'fj-stats';

    const msg = document.createElement('div');
    msg.className = 'fj-msg';
    msg.style.marginTop = '6px';
    msg.style.color = 'rgba(234,240,255,0.9)';

    hud.appendChild(title);
    hud.appendChild(stats);
    hud.appendChild(msg);
    container.appendChild(hud);

    this._hud = hud;
    this.renderHudStats();
  }

  renderHudStats() {
    if (!this._hud) return;
    const stats = this._hud.querySelector('.fj-stats');
    if (!stats) return;

    const nextJump = this.jumpIndex < this.fiboSeq.length ? this.fiboSeq[this.jumpIndex] : '—';

    stats.textContent =
      'Position: ' + this.units + ' / ' + this.targetUnits +
      '   |   Next jump: ' + nextJump +
      '   |   Moves left: ' + (this.fiboSeq.length - this.jumpIndex);
  }

  updateHud(message) {
    if (!this._hud) return;
    const msg = this._hud.querySelector('.fj-msg');
    if (msg) msg.textContent = message;
    this.renderHudStats();
  }

  showModal(title, body, tone) {
    const container = this.gameEnv && this.gameEnv.container ? this.gameEnv.container : null;
    if (!container) return;

    this.hideModal();

    const overlay = document.createElement('div');
    overlay.className = 'fj-modal';
    overlay.style.position = 'absolute';
    overlay.style.inset = '0';
    overlay.style.zIndex = '10000';
    overlay.style.display = 'flex';
    overlay.style.alignItems = 'center';
    overlay.style.justifyContent = 'center';
    overlay.style.background = 'rgba(0,0,0,0.55)';

    const card = document.createElement('div');
    card.style.width = 'min(520px, 92%)';
    card.style.borderRadius = '16px';
    card.style.padding = '14px 16px';
    card.style.border = '1px solid rgba(255,255,255,0.20)';
    card.style.background = 'rgba(10,14,24,0.92)';
    card.style.color = '#EAF0FF';
    card.style.fontFamily = 'ui-sans-serif, system-ui, -apple-system, Segoe UI, sans-serif';

    const h = document.createElement('div');
    h.style.fontWeight = '900';
    h.style.fontSize = '16px';
    h.style.marginBottom = '8px';
    h.style.color = tone === 'win' ? '#4CFFB5' : tone === 'lose' ? '#FF4C7E' : '#EAF0FF';
    h.textContent = title;

    const p = document.createElement('div');
    p.style.fontSize = '13px';
    p.style.color = 'rgba(234,240,255,0.9)';
    p.style.marginBottom = '12px';
    p.textContent = body;

    const hint = document.createElement('div');
    hint.style.fontSize = '12px';
    hint.style.color = 'rgba(234,240,255,0.75)';
    hint.textContent = 'Tip: click Stop then Play to restart, or switch levels and back.';

    card.appendChild(h);
    card.appendChild(p);
    card.appendChild(hint);
    overlay.appendChild(card);
    container.appendChild(overlay);

    this._modal = overlay;
  }

  hideModal() {
    if (this._modal && this._modal.parentNode) this._modal.parentNode.removeChild(this._modal);
    this._modal = null;
  }

  handleKeyDown({ keyCode }) {
    if (this.done) return;
    if (this._lastKeyDown === keyCode) return; // block OS key repeat
    this._lastKeyDown = keyCode;

    // Soft reset (keeps runner working)
    if (keyCode === 82) { // R
      this.resetRun();
      return;
    }

    if (this.jumpIndex >= this.fiboSeq.length) return;

    let dir = 0;
    if (keyCode === this.keypress.right) dir = 1;
    if (keyCode === this.keypress.left) dir = -1;
    if (dir === 0) return;

    const jumpVal = this.fiboSeq[this.jumpIndex++];
    this.units += dir * jumpVal;

    // GameEngine renders from position
    this.position.x = this.startX + (this.units * this.scalar);

    if (this.units === this.targetUnits) {
      this.done = true;
      this.updateHud('Win! You hit 33 exactly.');
      this.showModal('YOU WIN', 'You landed exactly on 33 using Fibonacci jumps.', 'win');
      return;
    }

    if (this.units > this.safeMax || this.units < this.safeMin) {
      this.done = true;
      this.updateHud('Out of bounds.');
      this.showModal('GAME OVER', 'You went out of bounds (' + this.safeMin + ' to ' + this.safeMax + ').', 'lose');
      return;
    }

    const dirWord = dir > 0 ? 'RIGHT' : 'LEFT';
    this.updateHud('Jumped ' + dirWord + ' by ' + jumpVal + '.');
  }

  resetRun() {
    this.done = false;
    this.jumpIndex = 0;
    this.units = 0;
    this.position.x = this.startX;
    this.hideModal();
    this.updateHud('Reset! Press A/D to jump again.');
  }

  handleKeyUp({ keyCode }) {
    if (this._lastKeyDown === keyCode) this._lastKeyDown = null;
  }
}

class FibonacciLevel {
  constructor(gameEnv) {
    const path = gameEnv.path;
    const width = gameEnv.innerWidth;
    const height = gameEnv.innerHeight;

    const bgData = {
      name: 'fibonacci_bg',
      src: path + '/images/gamebuilder/bg/alien_planet.jpg',
      pixels: { height: 720, width: 1280 }
    };

    const playerData = {
      id: 'Jumper',
      src: path + '/images/gamify/chillguy.png',
      SCALE_FACTOR: 5,
      STEP_FACTOR: 1000,
      ANIMATION_RATE: 50,
      INIT_POSITION: { x: width * 0.05, y: height * 0.72 },
      pixels: { height: 512, width: 384 },
      orientation: { rows: 4, columns: 3 },
      down: { row: 0, start: 0, columns: 3 },
      downRight: { row: 1, start: 0, columns: 3, rotate: Math.PI / 16 },
      downLeft: { row: 2, start: 0, columns: 3, rotate: -Math.PI / 16 },
      right: { row: 1, start: 0, columns: 3 },
      left: { row: 2, start: 0, columns: 3 },
      up: { row: 3, start: 0, columns: 3 },
      upRight: { row: 1, start: 0, columns: 3, rotate: -Math.PI / 16 },
      upLeft: { row: 2, start: 0, columns: 3, rotate: Math.PI / 16 },
      hitbox: { widthPercentage: 0.45, heightPercentage: 0.2 },
      keypress: { up: 87, left: 65, down: 83, right: 68 }
    };

    function checkpoint(id, unitsFromStart, label, detail, x0, scalar, yPos) {
      return {
        id: id,
        greeting: label,
        dialogues: [
          label,
          detail,
          'Gameplay: your next jump must be the next Fibonacci number.'
        ],
        fillStyle: '#F6E05E',
        SCALE_FACTOR: 40,
        visible: true,
        INIT_POSITION: { x: x0 + (unitsFromStart * scalar), y: yPos },
        pixels: { height: 16, width: 16 },
        hitbox: { widthPercentage: 1.0, heightPercentage: 1.0 },
        interact: function() {
          if (this.dialogueSystem) this.showRandomDialogue();
          else console.log(this.greeting);
        }
      };
    }

    const x0 = width * 0.05;
    const scalar = width * 0.02;
    const yLane = height * 0.72;

    // Fibonacci cumulative-sum milestones to 33: 1,2,4,7,12,20,33
    const cp1 = checkpoint('F(1)', 1, 'Checkpoint: 1', 'F(1)=1. Fibonacci starts with 1, 1.', x0, scalar, yLane);
    const cp2 = checkpoint('F(2)', 2, 'Checkpoint: 2', '1 + 1 = 2. Each term is the sum of the previous two.', x0, scalar, yLane);
    const cp3 = checkpoint('F(3)', 4, 'Checkpoint: 4', '2 + 2? No: the next jump value is 2, but your position is the running total.', x0, scalar, yLane);
    const cp4 = checkpoint('F(4)', 7, 'Checkpoint: 7', 'Try to reason about totals, not just the next jump.', x0, scalar, yLane);
    const cp5 = checkpoint('F(5)', 12, 'Checkpoint: 12', 'Fibonacci jumps grow fast: 5, 8, 13...', x0, scalar, yLane);
    const cp6 = checkpoint('F(6)', 20, 'Checkpoint: 20', 'At 20, the next correct jumps are 13 to hit 33 exactly.', x0, scalar, yLane);
    const cp7 = checkpoint('TARGET', 33, 'TARGET: 33', 'Perfect run: 1+1+2+3+5+8+13 = 33. You win on exact.', x0, scalar, yLane);

    const rulesStation = {
      id: 'Rules Station',
      greeting: 'Controls: A left, D right, R reset. Interact near checkpoints with E.',
      dialogues: [
        'Fibonacci Jumper Rules',
        'Press A (left) / D (right). Each press uses the next Fibonacci number as your jump distance.',
        'Goal: land exactly on 33. Press R to reset your run.'
      ],
      fillStyle: '#4CFFB5',
      SCALE_FACTOR: 35,
      visible: true,
      INIT_POSITION: { x: width * 0.12, y: height * 0.40 },
      pixels: { height: 16, width: 16 },
      hitbox: { widthPercentage: 1.0, heightPercentage: 1.0 },
      interact: function() {
        if (this.dialogueSystem) this.showRandomDialogue();
        else console.log(this.greeting);
      }
    };

    const topBarrier = {
      id: 'top_wall',
      xPercentage: 0.5,
      yPercentage: 0.1,
      widthPercentage: 1.0,
      heightPercentage: 0.03
    };

    const bottomBarrier = {
      id: 'bottom_wall',
      xPercentage: 0.5,
      yPercentage: 0.95,
      widthPercentage: 1.0,
      heightPercentage: 0.03
    };

    this.classes = [
      { class: GameEnvBackground, data: bgData },
      { class: Barrier, data: topBarrier },
      { class: Barrier, data: bottomBarrier },
      { class: JumperPlayer, data: playerData },
      { class: NPC, data: rulesStation },
      { class: NPC, data: cp1 },
      { class: NPC, data: cp2 },
      { class: NPC, data: cp3 },
      { class: NPC, data: cp4 },
      { class: NPC, data: cp5 },
      { class: NPC, data: cp6 },
      { class: NPC, data: cp7 }
    ];
  }
}

export const gameLevelClasses = [FibonacciLevel];
export { GameControl };


In [ ]:
/*
 * Creator: Open Coding Society
 * Mini Lab Name: Fibonacci Jumper - Java Backend
 */

import java.util.ArrayList;

/* AP CSA Topic: Abstract Classes and Methods */
abstract class Game {
    String name;
    int currentPos;
    int targetPos;
    ArrayList<Integer> sequence; // AP CSA Topic: ArrayList

    public Game(String name, int targetPos) {
        this.name = name;
        this.targetPos = targetPos;
        this.currentPos = 0;
        this.sequence = new ArrayList<>();
    }

    // Abstract method to generate the sequence (AP CSA Topic: Arrays & Loops)
    protected abstract void generateSequence(int limit);

    // Method to jump (AP CSA Topic: Conditionals & Methods)
    public abstract boolean jump(int direction);
}


In [2]:
/* AP CSA Topic: Inheritance and Classes */
public class FibonacciJumper extends Game {
    private int jumpCount;

    public FibonacciJumper(int target) {
        super("Fibonacci Jumper", target);
        this.jumpCount = 0;
        generateSequence(15); // Pre-generate 15 potential jumps
    }

    @Override
    protected void generateSequence(int limit) {
        // AP CSA Topic: Iteration & Algorithms
        sequence.add(1);
        sequence.add(1);
        for(int i = 2; i < limit; i++) {
            sequence.add(sequence.get(i-1) + sequence.get(i-2));
        }
    }

    @Override
    public boolean jump(int direction) {
        // AP CSA Topic: Conditionals (if/else)
        if (jumpCount >= sequence.size()) {
            System.out.println("Out of jumps!");
            return false;
        }
        
        int jumpDist = sequence.get(jumpCount);
        if (direction > 0) {
            currentPos += jumpDist;
            System.out.println("Jumped RIGHT by " + jumpDist + ". Position: " + currentPos);
        } else if (direction < 0) {
            currentPos -= jumpDist;
            System.out.println("Jumped LEFT by " + jumpDist + ". Position: " + currentPos);
        }
        jumpCount++;

        return checkCondition();
    }

    private boolean checkCondition() {
        if (currentPos == targetPos) {
            System.out.println("🎉 YOU WIN! Landed exactly on " + targetPos + "!");
            return true;
        } else if (currentPos > targetPos + 20 || currentPos < -5) {
            System.out.println("💀 GAME OVER! Overshot the target into oblivion.");
            return true;
        }
        return false;
    }
}


Calculation method = FiboFor extends Fibo
fibonacci Number 2 = 1
fibonacci List = [0, 1]
fibonacci Hashmap = {0=[0], 1=[0, 1]}
fibonacci Sequence 1 = [0]
fibonacci Sequence 2 = [0, 1]

Calculation method = FiboFor extends Fibo
fibonacci Number 5 = 3
fibonacci List = [0, 1, 1, 2, 3]
fibonacci Hashmap = {0=[0], 1=[0, 1], 2=[0, 1, 1], 3=[0, 1, 1, 2], 4=[0, 1, 1, 2, 3]}
fibonacci Sequence 1 = [0]
fibonacci Sequence 2 = [0, 1]
fibonacci Sequence 3 = [0, 1, 1]
fibonacci Sequence 4 = [0, 1, 1, 2]
fibonacci Sequence 5 = [0, 1, 1, 2, 3]

Calculation method = FiboFor extends Fibo
fibonacci Number 8 = 13
fibonacci List = [0, 1, 1, 2, 3, 5, 8, 13]
fibonacci Hashmap = {0=[0], 1=[0, 1], 2=[0, 1, 1], 3=[0, 1, 1, 2], 4=[0, 1, 1, 2, 3], 5=[0, 1, 1, 2, 3, 5], 6=[0, 1, 1, 2, 3, 5, 8], 7=[0, 1, 1, 2, 3, 5, 8, 13]}
fibonacci Sequence 1 = [0]
fibonacci Sequence 2 = [0, 1]
fibonacci Sequence 3 = [0, 1, 1]
fibonacci Sequence 4 = [0, 1, 1, 2]
fibonacci Sequence 5 = [0, 1, 1, 2, 3]
fibonacci Sequence 6 = [0, 1,

In [3]:
/* AP CSA Topic: Main execution and Traced runs */
public class Tester {
    public static void main(String[] args) {
        // Target is 33
        // Optimal Jumps: 1(R)+1(R)+2(R)+3(R)+5(R)+8(R)+13(R) = 33
        FibonacciJumper game = new FibonacciJumper(33);
        
        System.out.println("Starting " + game.name + ". Target is " + game.targetPos);
        
        game.jump(1);  // +1
        game.jump(1);  // +1
        game.jump(1);  // +2
        game.jump(1);  // +3
        game.jump(1);  // +5
        game.jump(-1); // OH NO! Went left -8. Position should be 12 - 8 = 4
        game.jump(1);  // +13. Position 4 + 13 = 17.
        game.jump(1);  // +21. Position 17 + 21 = 38. OVER SHOT! (Target is 33)
    }
}
Tester.main(null);


Calculation method = FiboFor extends Fibo
fibonacci Number 2 = 1
fibonacci List = [0, 1]
fibonacci Hashmap = {0=[0], 1=[0, 1]}
fibonacci Sequence 1 = [0]
fibonacci Sequence 2 = [0, 1]

Calculation method = FiboFor extends Fibo
fibonacci Number 5 = 3
fibonacci List = [0, 1, 1, 2, 3]
fibonacci Hashmap = {0=[0], 1=[0, 1], 2=[0, 1, 1], 3=[0, 1, 1, 2], 4=[0, 1, 1, 2, 3]}
fibonacci Sequence 1 = [0]
fibonacci Sequence 2 = [0, 1]
fibonacci Sequence 3 = [0, 1, 1]
fibonacci Sequence 4 = [0, 1, 1, 2]
fibonacci Sequence 5 = [0, 1, 1, 2, 3]

Calculation method = FiboFor extends Fibo
fibonacci Number 8 = 13
fibonacci List = [0, 1, 1, 2, 3, 5, 8, 13]
fibonacci Hashmap = {0=[0], 1=[0, 1], 2=[0, 1, 1], 3=[0, 1, 1, 2], 4=[0, 1, 1, 2, 3], 5=[0, 1, 1, 2, 3, 5], 6=[0, 1, 1, 2, 3, 5, 8], 7=[0, 1, 1, 2, 3, 5, 8, 13]}
fibonacci Sequence 1 = [0]
fibonacci Sequence 2 = [0, 1]
fibonacci Sequence 3 = [0, 1, 1]
fibonacci Sequence 4 = [0, 1, 1, 2]
fibonacci Sequence 5 = [0, 1, 1, 2, 3]
fibonacci Sequence 6 = [0, 1,

## Popcorn Hacks
Objectives of these hacks are ...

1. Understand how to fullfill abstract class requirements using two additional algoritms.
2. Use inheritance style of programming to test speed of each algorithm.  To test the speed, a.) be aware that the first run is always the slowest b.) to time something, my recommendation is 12 runs on the timed element, through out highest and lowest time in calculations.
3. Be sure to make a tester and reporting methods.

.85 basis for text based comparison inside of Jupyter Notebook lesson

## Hacks
Assign in each Team to build a Thymeleaf UI for pages using this example https://thymeleaf.nighthawkcodingsociety.com/mvc/fibonacci as basis.  Encorporate into Algorithms menu.

Since there are three teams, one team can do Fibo, others Pali and Factorial.  Assign this to people that are struggling for contribution and presentation to checkpoints.

.90 basis for FE presentation in Thymmeleaf to BE call in Spring